# FiQA Cross-Dataset Validation (GPU)

**Purpose**: Encode FiQA 57K docs on Colab T4 GPU, run Riverbed×Tension vs RRF vs Dense-only.

**Output**: `cross_dataset_fiqa.json` + `.cache_fiqa_e5pt_base_embs.npz` (download both)

In [3]:
!pip install -q beir sentence-transformers rank-bm25 numpy
!nvidia-smi

Fri Mar 27 11:37:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             16W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import json
import math
import os
import re
import time
from collections import defaultdict

import numpy as np
import torch
from beir import util
from beir.datasets.data_loader import GenericDataLoader
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"VRAM: {vram / 1e9:.1f} GB")

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


In [5]:
# ── Fixed parameters (same as local cross_dataset_validate.py) ──

FIXED_PARAMS = {
    "k_low": 3, "k_high": 10, "top_n": 20,
    "boost_max": 1.2, "score_w": 0.5, "bw": 0.8, "dw": 1.4,
}

BASELINE_RRF_PARAMS = {"k": 5, "bw": 1.0, "dw": 1.2}

# Also test universal params
UNIVERSAL_PARAMS = {
    "k_low": 2, "k_high": 5, "top_n": 20,
    "boost_max": 1.2, "score_w": 0.5, "bw": 0.8, "dw": 1.0,
}

print("E5PT params:", FIXED_PARAMS)
print("Universal params:", UNIVERSAL_PARAMS)
print("RRF baseline:", BASELINE_RRF_PARAMS)

E5PT params: {'k_low': 3, 'k_high': 10, 'top_n': 20, 'boost_max': 1.2, 'score_w': 0.5, 'bw': 0.8, 'dw': 1.4}
Universal params: {'k_low': 2, 'k_high': 5, 'top_n': 20, 'boost_max': 1.2, 'score_w': 0.5, 'bw': 0.8, 'dw': 1.0}
RRF baseline: {'k': 5, 'bw': 1.0, 'dw': 1.2}


In [6]:
# ── Fusion functions ──

def simple_rrf(b, d, k=5, bw=1.0, dw=1.2):
    scores = defaultdict(float)
    for rank, (did, _) in enumerate(b):
        scores[did] += bw / (k + rank + 1)
    for rank, (did, _) in enumerate(d):
        scores[did] += dw / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def riverbed_tension(b, d, k_low=2, k_high=10, top_n=10,
                     boost_max=1.3, score_w=0.3, bw=1.0, dw=1.2):
    b_set = {did for did, _ in b[:top_n]}
    d_set = {did for did, _ in d[:top_n]}
    union = b_set | d_set
    agreement = len(b_set & d_set) / len(union) if union else 0.0
    tension = 1.0 - agreement
    adaptive_k = max(1, int(k_low + (k_high - k_low) * tension))
    boost = 1.0 + (boost_max - 1.0) * agreement

    rrf_scores = defaultdict(float)
    presence = defaultdict(int)
    for rank, (did, _) in enumerate(b):
        rrf_scores[did] += bw / (adaptive_k + rank + 1)
        presence[did] += 1
    for rank, (did, _) in enumerate(d):
        rrf_scores[did] += dw / (adaptive_k + rank + 1)
        presence[did] += 1
    for did in rrf_scores:
        if presence[did] >= 2:
            rrf_scores[did] *= boost

    def norm(results):
        if not results:
            return {}
        vals = [s for _, s in results]
        mn, mx = min(vals), max(vals)
        rng = mx - mn if mx > mn else 1.0
        return {did: (s - mn) / rng for did, s in results}

    b_n, d_n = norm(b), norm(d)
    rv = list(rrf_scores.values())
    r_mn, r_mx = min(rv), max(rv)
    r_rng = r_mx - r_mn if r_mx > r_mn else 1.0

    all_docs = set(rrf_scores) | set(b_n) | set(d_n)
    final = {}
    for did in all_docs:
        r = (rrf_scores.get(did, 0) - r_mn) / r_rng
        s = (bw * b_n.get(did, 0) + dw * d_n.get(did, 0)) / (bw + dw)
        final[did] = (1 - score_w) * r + score_w * s
    return sorted(final.items(), key=lambda x: x[1], reverse=True)


def dense_only_passthrough(b, d):
    return d

print("Fusion functions ready")

Fusion functions ready


In [7]:
# ── Metrics ──

def ndcg_at_k(ranked_ids, relevant, k=10):
    dcg = sum(relevant.get(did, 0) / math.log2(i + 2)
              for i, did in enumerate(ranked_ids[:k]))
    ideal = sorted(relevant.values(), reverse=True)[:k]
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0

def recall_at_k(ranked_ids, relevant, k=100):
    if not relevant:
        return 0.0
    return sum(1 for did in ranked_ids[:k] if did in relevant) / len(relevant)

print("Metrics ready")

Metrics ready


In [8]:
# ── Download FiQA + Load ──

FIQA_URL = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/fiqa.zip"
BASE_DIR = "datasets"
os.makedirs(BASE_DIR, exist_ok=True)

data_path = os.path.join(BASE_DIR, "fiqa")
if not os.path.isdir(data_path):
    print("Downloading FiQA...")
    data_path = util.download_and_unzip(FIQA_URL, BASE_DIR)

corpus, queries, qrels = GenericDataLoader(data_path).load(split="test")
print(f"Corpus: {len(corpus)} docs | Queries: {len(queries)}")

datasets/fiqa.zip:   0%|          | 0.00/17.1M [00:00<?, ?iB/s]

  0%|          | 0/57638 [00:00<?, ?it/s]

Corpus: 57638 docs | Queries: 648


In [9]:
# ── Load model on GPU ──

print("Loading E5-PT base on GPU...")
model = SentenceTransformer("intfloat/e5-base-unsupervised", device=DEVICE)
print(f"Model loaded on {DEVICE}")

Loading E5-PT base on GPU...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Model loaded on cuda


In [10]:
# ── Encode corpus with checkpoint ──

CACHE_PATH = os.path.join(BASE_DIR, ".cache_fiqa_e5pt_base_embs.npz")
CKPT_PATH = CACHE_PATH + ".ckpt.npz"

doc_id_list = list(corpus.keys())
texts = []
for doc_id in doc_id_list:
    doc = corpus[doc_id]
    title = doc.get("title", "")
    text = doc.get("text", "")
    texts.append(f"passage: {title} {text}".strip())

print(f"Total texts to encode: {len(texts)}")

if os.path.exists(CACHE_PATH):
    print(f"Cache exists, loading: {CACHE_PATH}")
    data = np.load(CACHE_PATH)
    passage_embs = data["embs"]
    print(f"Loaded {passage_embs.shape}")
else:
    start_idx = 0
    all_embs = []

    # Resume from checkpoint
    if os.path.exists(CKPT_PATH):
        ckpt = np.load(CKPT_PATH)
        start_idx = int(ckpt["done"])
        all_embs = [ckpt["embs"]]
        print(f"Resuming from checkpoint: {start_idx}/{len(texts)}")

    batch_size = 256  # GPU can handle larger batches
    t0 = time.time()
    for i in range(start_idx, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        embs = model.encode(batch, normalize_embeddings=True,
                            show_progress_bar=False, batch_size=128)
        all_embs.append(embs)
        done = min(i + batch_size, len(texts))
        # Checkpoint every 5000 docs
        if done % 5000 < batch_size or done == len(texts):
            partial = np.vstack(all_embs)
            np.savez_compressed(CKPT_PATH, embs=partial, done=done)
            elapsed = time.time() - t0
            speed = (done - start_idx) / elapsed if elapsed > 0 else 0
            remaining = (len(texts) - done) / speed if speed > 0 else 0
            print(f"  {done}/{len(texts)} ({elapsed:.0f}s, {speed:.0f} docs/s, ETA {remaining:.0f}s)")

    passage_embs = np.vstack(all_embs)
    np.savez_compressed(CACHE_PATH, embs=passage_embs)
    if os.path.exists(CKPT_PATH):
        os.remove(CKPT_PATH)
    print(f"\nDone! {passage_embs.shape} in {time.time() - t0:.1f}s")
    print(f"Saved: {CACHE_PATH}")

Total texts to encode: 57638
  5120/57638 (109s, 47 docs/s, ETA 1118s)
  10240/57638 (227s, 45 docs/s, ETA 1050s)
  15104/57638 (343s, 44 docs/s, ETA 967s)
  20224/57638 (462s, 44 docs/s, ETA 855s)
  25088/57638 (577s, 43 docs/s, ETA 748s)
  30208/57638 (702s, 43 docs/s, ETA 637s)
  35072/57638 (822s, 43 docs/s, ETA 529s)
  40192/57638 (949s, 42 docs/s, ETA 412s)
  45056/57638 (1070s, 42 docs/s, ETA 299s)
  50176/57638 (1195s, 42 docs/s, ETA 178s)
  55040/57638 (1315s, 42 docs/s, ETA 62s)
  57638/57638 (1384s, 42 docs/s, ETA 0s)

Done! (57638, 768) in 1393.7s
Saved: datasets/.cache_fiqa_e5pt_base_embs.npz


In [11]:
# ── BM25 Index ──

def tokenize(text):
    return re.findall(r'\w+', text.lower())

bm25_ids = list(corpus.keys())
tokenized = [tokenize(f"{corpus[did].get('title', '')} {corpus[did].get('text', '')}") for did in bm25_ids]

t0 = time.time()
bm25 = BM25Okapi(tokenized)
print(f"BM25 built in {time.time() - t0:.1f}s ({len(bm25_ids)} docs)")

def search_bm25(query, top_k=100):
    scores = bm25.get_scores(tokenize(query))
    top_idx = scores.argsort()[-top_k:][::-1]
    return [(bm25_ids[i], float(scores[i])) for i in top_idx if scores[i] > 0]

BM25 built in 2.1s (57638 docs)


In [12]:
# ── Dense search function ──

def dense_search(query_text, top_k=100):
    q = model.encode(["query: " + query_text], normalize_embeddings=True)
    sims = (passage_embs @ q.T).flatten()
    idx = np.argsort(sims)[::-1][:top_k]
    return [(doc_id_list[i], float(sims[i])) for i in idx]

print("Dense search ready")

Dense search ready


In [13]:
# ── Cache all queries ──

print(f"Caching {len(queries)} query results...")
cached = {}
t0 = time.time()
for qi, (qid, qt) in enumerate(queries.items()):
    rel = {d: r for d, r in qrels.get(qid, {}).items() if r > 0}
    cached[qid] = {
        "bm25": search_bm25(qt, top_k=100),
        "dense": dense_search(qt, top_k=100),
        "rel": rel,
    }
    if (qi + 1) % 100 == 0:
        print(f"  {qi+1}/{len(queries)}")
print(f"Cached {len(cached)} queries in {time.time() - t0:.1f}s")

Caching 648 query results...
  100/648
  200/648
  300/648
  400/648
  500/648
  600/648
Cached 648 queries in 193.3s


In [14]:
# ── Evaluate all strategies ──

def eval_strategy(fusion_fn, **kw):
    ndcgs, recalls = [], []
    for e in cached.values():
        fused = fusion_fn(e["bm25"], e["dense"], **kw)
        ranked = [d for d, _ in fused[:100]]
        ndcgs.append(ndcg_at_k(ranked, e["rel"], k=10))
        recalls.append(recall_at_k(ranked, e["rel"], k=100))
    return round(sum(ndcgs)/len(ndcgs), 4), round(sum(recalls)/len(recalls), 4)

print("="*60)
print("FiQA Results")
print("="*60)

# Dense only
dense_ndcg, dense_recall = eval_strategy(dense_only_passthrough)
print(f"Dense only:        nDCG@10={dense_ndcg:.4f}  R@100={dense_recall:.4f}")

# Simple RRF
rrf_ndcg, rrf_recall = eval_strategy(simple_rrf, **BASELINE_RRF_PARAMS)
print(f"Simple RRF:        nDCG@10={rrf_ndcg:.4f}  R@100={rrf_recall:.4f}")

# RT E5PT params
rt_ndcg, rt_recall = eval_strategy(riverbed_tension, **FIXED_PARAMS)
print(f"RT (E5PT):         nDCG@10={rt_ndcg:.4f}  R@100={rt_recall:.4f}")

# RT Universal params
rtu_ndcg, rtu_recall = eval_strategy(riverbed_tension, **UNIVERSAL_PARAMS)
print(f"RT (Universal):    nDCG@10={rtu_ndcg:.4f}  R@100={rtu_recall:.4f}")

print(f"\nRT(E5PT) vs Dense:     {rt_ndcg - dense_ndcg:+.4f}")
print(f"RT(Universal) vs Dense: {rtu_ndcg - dense_ndcg:+.4f}")
print(f"RRF vs Dense:           {rrf_ndcg - dense_ndcg:+.4f}")
print(f"RT(E5PT) vs RRF:        {rt_ndcg - rrf_ndcg:+.4f}")
print(f"RT(Universal) vs RRF:   {rtu_ndcg - rrf_ndcg:+.4f}")

FiQA Results
Dense only:        nDCG@10=0.4008  R@100=0.7176
Simple RRF:        nDCG@10=0.3955  R@100=0.7071
RT (E5PT):         nDCG@10=0.4122  R@100=0.7147
RT (Universal):    nDCG@10=0.4000  R@100=0.7084

RT(E5PT) vs Dense:     +0.0114
RT(Universal) vs Dense: -0.0008
RRF vs Dense:           -0.0053
RT(E5PT) vs RRF:        +0.0167
RT(Universal) vs RRF:   +0.0045


In [15]:
# ── Save results ──

output = {
    "dataset": "fiqa",
    "corpus_size": len(corpus),
    "num_queries": len(queries),
    "model": "intfloat/e5-base-unsupervised",
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU",
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "fixed_params_e5pt": FIXED_PARAMS,
    "fixed_params_universal": UNIVERSAL_PARAMS,
    "baseline_rrf_params": BASELINE_RRF_PARAMS,
    "results": {
        "dense_only": {"ndcg@10": dense_ndcg, "recall@100": dense_recall},
        "simple_rrf": {"ndcg@10": rrf_ndcg, "recall@100": rrf_recall},
        "rt_e5pt": {"ndcg@10": rt_ndcg, "recall@100": rt_recall},
        "rt_universal": {"ndcg@10": rtu_ndcg, "recall@100": rtu_recall},
    },
    "deltas": {
        "rt_e5pt_vs_dense": round(rt_ndcg - dense_ndcg, 4),
        "rt_universal_vs_dense": round(rtu_ndcg - dense_ndcg, 4),
        "rrf_vs_dense": round(rrf_ndcg - dense_ndcg, 4),
        "rt_e5pt_vs_rrf": round(rt_ndcg - rrf_ndcg, 4),
        "rt_universal_vs_rrf": round(rtu_ndcg - rrf_ndcg, 4),
    },
    "cross_dataset_summary": {
        "scifact": {"rt": 0.7578, "rrf": 0.7541, "dense": 0.7371},
        "nfcorpus": {"rt": 0.3680, "rrf": 0.3631, "dense": 0.3594},
        "fiqa": {"rt_e5pt": rt_ndcg, "rt_universal": rtu_ndcg, "rrf": rrf_ndcg, "dense": dense_ndcg},
    },
}

with open("cross_dataset_fiqa.json", "w") as f:
    json.dump(output, f, indent=2)

print("Saved: cross_dataset_fiqa.json")
print("\n" + "="*60)
print("DOWNLOAD THESE FILES:")
print("="*60)
print(f"1. cross_dataset_fiqa.json")
print(f"2. {CACHE_PATH}")
print("\nPut them in: E:/Cursor/projects/tempero/benchmarks/")
print(f"  - JSON -> results/cross_dataset_fiqa.json")
print(f"  - NPZ  -> datasets/.cache_fiqa_e5pt_base_embs.npz")

Saved: cross_dataset_fiqa.json

DOWNLOAD THESE FILES:
1. cross_dataset_fiqa.json
2. datasets/.cache_fiqa_e5pt_base_embs.npz

Put them in: E:/Cursor/projects/tempero/benchmarks/
  - JSON -> results/cross_dataset_fiqa.json
  - NPZ  -> datasets/.cache_fiqa_e5pt_base_embs.npz


In [ ]:
# ── Create download link via colab file server ──
# Method: base64 encode the JSON (small), print npz size info
import base64

# JSON is small, embed directly
with open("cross_dataset_fiqa.json", "r") as f:
    json_content = f.read()
b64 = base64.b64encode(json_content.encode()).decode()
print("=== COPY THIS JSON (paste to Claude) ===")
print(json_content)
print("=== END JSON ===")

# NPZ is large, show size and provide alternative
npz_size = os.path.getsize(CACHE_PATH) / 1e6
print(f"\nNPZ cache: {CACHE_PATH} ({npz_size:.1f} MB)")
print("NPZ too large for copy-paste.")
print("\nTo get NPZ, run this in a NEW Colab cell:")
print("  !cp", CACHE_PATH, "/content/")
print("  # Then use Colab web UI: Files panel (left) -> right-click -> Download")
print("\nOR just re-run this notebook next time you need FiQA embeddings.")